# t-SNE／UMAP（2008／2018）

- **核心原理**：保留局部鄰近結構的非線性降維
- **變體**：Barnes-Hut t-SNE（$O(n \log n)$ 近似）、UMAP（模糊鄰居圖 + 交叉熵，速度快、可以 `transform` 新資料）
- **推薦資料集**：MNIST（70,000 張 28×28 手寫數字；本章取 **5,000 張子集**，因為手寫 t-SNE 要算 $n \times n$ 的矩陣）

第十六章的 PCA 是**線性**的：只能沿著直線方向壓縮，目標是「保留最多變異」。結果 Digits 的 PCA 2D 圖上，1、7、8、5、2 擠成一團（第四章、第十六章 EDA）。

t-SNE 換一個目標：**不在乎變異，只在乎「誰是誰的鄰居」**。

1. 在高維空間，把「$j$ 是 $i$ 的鄰居的機率」寫成 $p_{ij}$（近的大、遠的幾乎 0）；
2. 在 2 維空間，同樣寫一個 $q_{ij}$；
3. 移動 2 維的點，讓 $q$ 盡量像 $p$（最小化 KL 散度）。

UMAP 的想法相同（鄰居圖），但用不同的機率定義與損失，速度快很多。

這一章的重點：**perplexity（每個點「看」幾個鄰居）與二分搜尋找 $\sigma_i$、Student-t 分布解決「擁擠問題」、KL 的梯度 = 吸引力 − 排斥力、early exaggeration、為什麼 t-SNE 圖上的「群的大小」和「群之間的距離」沒有意義、UMAP 的超參數，以及怎麼評估一張降維圖（trustworthiness、鄰居重疊率）。**

### 閱讀指南：本章的符號與跨章引用

下一格是「前置複習」；第 2、3A、3B、3C、4、5 節開頭另有各自的符號表或名詞表。

**資料**：MNIST 每張影像 28×28 = 784 個像素（灰階 0–255），攤平成長度 784 的向量。先用 PCA 降到 50 維（第十六章）再做 t-SNE —— 去掉雜訊、算距離快 15 倍。標籤 0–9 **只用來上色與評估**，t-SNE 從頭到尾沒看過。另外用第十六章的 Digits（1,797 張 8×8）當「快速驗證」用的小資料。

**本章最常出現的符號**（各小節開頭有完整符號表）：

| 符號 | 意義 | 本章的值 |
| --- | --- | --- |
| $n$、$D$ | 樣本數、高維的維度 | MNIST 5,000、50（PCA 後）；Digits 1,797、64 |
| $x_i$、$y_i$ | 第 $i$ 個點的高維座標、低維（2 維）座標 | $(D,)$、$(2,)$ |
| $d_{ij}$ | 高維平方距離 $\lVert x_i - x_j\rVert^2$ | $(n, n)$ |
| $\sigma_i$、$\beta_i$ | 點 $i$ 的高斯寬度；$\beta_i = 1 / (2\sigma_i^2)$ | 每個點不同 |
| $p_{j\mid i}$ | 條件機率：$i$ 選 $j$ 當鄰居 | 每一列加總 1 |
| $p_{ij}$ | 對稱的聯合機率 $(p_{j\mid i} + p_{i\mid j}) / (2n)$ | 全部加總 1 |
| $q_{ij}$ | 低維的聯合機率（Student-t） | 全部加總 1 |
| Perp | perplexity $= e^{H(P_i)}$，「有效鄰居數」 | 預設 30 |
| $C$ | 損失 $\text{KL}(P\Vert Q) = \sum_{ij} p_{ij}\log(p_{ij}/q_{ij})$ | — |

**跨章引用對照**：

| 文中寫法 | 指的是 | 那裡的結論（重述） |
| --- | --- | --- |
| 第四章 | KNN（`01-classic-ml/04-knn`），Digits | 鄰居 = 距離最近的 K 個點；維度詛咒：高維時「最近」與「最遠」的距離差不多。 |
| 第六章 | 樸素貝氏 | 熵 $H = -\sum p\log p$；KL 散度 $\text{KL}(P\Vert Q) = \sum p \log(p/q) \ge 0$，$P = Q$ 時才等於 0。 |
| 第八章 | SVM | RBF kernel $\exp(-\gamma\lVert x - x'\rVert^2)$：t-SNE 高維的相似度就是它（每個點有自己的 $\gamma = \beta_i$）。 |
| 第十四章 | DBSCAN | 不同區域密度不同時，單一 eps 不夠用 —— t-SNE 讓每個點有自己的 $\sigma_i$ 解決同樣的問題。 |
| 第十六章 | PCA（`01-classic-ml/16-pca`），Digits | 線性、保留變異；2 維只保留 28.5% 的變異；trustworthiness（k = 2）0.830；成績在 `results/digits_embedding.csv`。 |

### 前置複習：本章會用到的舊知識

#### 數學

**1. 高斯相似度**：$\exp(-d / (2\sigma^2)) = \exp(-\beta d)$（$d$ 是**平方**距離）。$\sigma$ 大 → 遠的點也有份；$\sigma$ 小 → 只有最近的幾個點有份。例：$d = (1, 2, 4)$、$\beta = 1$ → $(e^{-1}, e^{-2}, e^{-4}) = (0.368, 0.135, 0.018)$，正規化後 $(0.705, 0.260, 0.035)$。

**2. 熵與 perplexity**（第六章）：$H(P_i) = -\sum_j p_{j\mid i}\ln p_{j\mid i}$（本章用自然對數）。perplexity $= e^{H}$ = 「等效於在幾個點裡平均挑一個」。例：4 個點均分 → $H = \ln 4$，perplexity = 4；只集中在 1 個點 → perplexity = 1。

**3. KL 散度**（第六章）：$\text{KL}(P\Vert Q) = \sum p\ln(p/q) \ge 0$。**不對稱**：$p$ 大、$q$ 小 → 懲罰很大（高維的鄰居在低維被拆散）；$p$ 小、$q$ 大 → 懲罰小（高維不相鄰的點在低維靠近）。所以 t-SNE **重視保留局部**，不太管遠的點。

**4. Student-t（自由度 1，又叫 Cauchy）**：$(1 + d)^{-1}$。跟高斯 $e^{-d}$ 比，**尾巴很重**：$d = 9$ 時高斯 $1.2 \times 10^{-4}$、t 是 0.1。

**5. 二分搜尋**：$f(\beta)$ 單調時，找 $f(\beta) = $ 目標值：太大就往一邊縮、太小往另一邊。這裡 $H(\beta)$ 隨 $\beta$ **遞減**（$\beta$ 越大越集中、熵越小）。

**6. 梯度下降 + momentum**：$\Delta_t = \mu\Delta_{t-1} - \eta\, g_t$，$y \leftarrow y + \Delta_t$。$\mu$ 是動量（0.5 → 0.8）、$\eta$ 是學習率。

#### 程式

**7. 平方距離矩陣**：$\lVert a - b\rVert^2 = \lVert a\rVert^2 + \lVert b\rVert^2 - 2a\cdot b$ → `s = (X**2).sum(1); s[:, None] + s[None, :] - 2 * X @ X.T`；浮點誤差可能產生 −1e-12，用 `np.maximum(D, 0)` 修掉。
**8. `np.fill_diagonal(A, 0)`**：把對角線（自己跟自己）設成 0。
**9. 減掉最小值再取 exp**：`np.exp(-(d - d.min()) * beta)` 跟 `np.exp(-d * beta)` 正規化後一樣，但不會全部 underflow 成 0（第六章 log-sum-exp 的同一招）。
**10. 廣播算「所有配對的差」**：`Y[:, None, :] - Y[None, :, :]` 是 $(n, n, 2)$；但更省記憶體的是矩陣乘法：$\sum_j W_{ij}(y_i - y_j) = (\text{diag}(W\mathbf 1) - W)\,Y$。

---
## 1. 資料與探索性資料分析（EDA）

MNIST 從 70,000 張裡隨機抽 5,000 張。看：類別平衡、影像長什麼樣子、跟 Digits 比解析度、PCA 降到 50 維保留了多少變異，以及 PCA 2D 的圖（第十六章的重述）。

In [ ]:
import time

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from ml_dl_llm.data import load_digits, load_mnist_784

X_df, y_ser = load_mnist_784(n_samples=5_000)
X_raw = np.asarray(X_df, dtype=float)
y = np.asarray(y_ser).astype(int)
n = len(X_raw)
print("X：", X_raw.shape, "；像素範圍：", X_raw.min(), "–", X_raw.max(), "；每一類：", np.bincount(y).tolist())

fig, axes = plt.subplots(2, 10, figsize=(14, 3))
for c in range(10):
    for r in range(2):
        axes[r, c].imshow(X_raw[np.flatnonzero(y == c)[r]].reshape(28, 28), cmap="gray_r")
        axes[r, c].axis("off")
plt.show()

pca50 = PCA(50, random_state=0).fit(X_raw)
X = pca50.transform(X_raw)  # 之後的 t-SNE / UMAP 都用這 50 維
print(f"PCA 50 維保留的變異：{pca50.explained_variance_ratio_.sum():.3f}；全為 0 的像素：{int((X_raw.var(axis=0) == 0).sum())} 個")

Z_pca = X[:, :2]
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(Z_pca[:, 0], Z_pca[:, 1], c=y, cmap="tab10", s=3)
for c in range(10):
    ax.text(*np.median(Z_pca[y == c], axis=0), str(c), fontsize=14, weight="bold")
ax.set_title("MNIST 5k: PCA 2D")
plt.show()

In [ ]:
# 高維距離的分布：「最近鄰居」與「隨便一個點」差多少？（第四章維度詛咒）
from sklearn.neighbors import NearestNeighbors

dist_all = np.sqrt(((X[:500, None, :] - X[None, :, :]) ** 2).sum(-1))
nn_d = NearestNeighbors(n_neighbors=31).fit(X).kneighbors(X[:500])[0]
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(dist_all[dist_all > 0].ravel(), bins=80, alpha=0.6, density=True, label="all pairs")
ax.hist(nn_d[:, 1], bins=40, alpha=0.6, density=True, label="1st nearest neighbor")
ax.hist(nn_d[:, 30], bins=40, alpha=0.6, density=True, label="30th nearest neighbor")
ax.set_xlabel("Euclidean distance (PCA-50 space)")
ax.legend()
plt.show()
print(f"中位數：最近鄰 {np.median(nn_d[:, 1]):.0f}、第 30 近 {np.median(nn_d[:, 30]):.0f}、所有配對 {np.median(dist_all[dist_all > 0]):.0f}")

In [ ]:
# TODO 1（問答）EDA 小結
#
# (1) MNIST 和 Digits 比：解析度、像素範圍、樣本數各差多少？5,000 張的 t-SNE 要存幾個 n × n 的 float64 矩陣？一個多大（MB）？
# 答：
#
# (2) PCA 2D 圖：哪些數字分得開、哪些擠在一起？跟第十六章 Digits 的 PCA 2D 圖比，像不像？
# 答：
#
# (3) 距離分布：「第 30 近的鄰居」和「隨便一個點」的距離差多少倍？
#     如果 2 維要「保留所有距離」，可能嗎？（提示：50 維裡可以有 51 個兩兩等距的點，2 維最多 3 個）
#     → 所以 t-SNE 只保留「誰比較近」，放棄保留遠距離。
# 答：

---
## 2. Baseline：線性降維（PCA 2D）與隨機投影

先量「一張 2D 圖保留了多少鄰居關係」：

- **trustworthiness**（`ml_dl_llm.metrics`，0–1）：低維的 5 個最近鄰居，在高維中也排得很前面的程度。低維有「假鄰居」就扣分。
- **2D 的 KNN 準確率**：只用 2 維座標做 5-NN 分類（5-fold CV）。標籤沒參與降維，所以準確率高 = 同一個數字在 2D 上聚在一起。

#### 本節名詞

| 名詞 | 意思 |
| --- | --- |
| `trustworthiness(X_high, X_low, n_neighbors=5)` | 回傳 `{"trustworthiness": ...}` |
| `knn_acc_2d(Z, y)` | 本章的小工具：2D 座標上的 5-NN 5-fold CV accuracy |

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from ml_dl_llm.metrics import trustworthiness

cv = StratifiedKFold(5, shuffle=True, random_state=42)


def knn_acc_2d(Z: np.ndarray, y: np.ndarray) -> float:
    """只用低維座標做 5-NN 分類的 CV accuracy。"""
    return float(cross_val_score(KNeighborsClassifier(5), Z, y, cv=cv).mean())


Q_rand, _ = np.linalg.qr(np.random.default_rng(0).normal(size=(X.shape[1], 2)))
baselines = {"PCA 2D": Z_pca, "Random projection 2D": (X - X.mean(axis=0)) @ Q_rand}
base_tbl = pd.DataFrame({name: {**trustworthiness(X, Z), "knn_acc_2d": knn_acc_2d(Z, y)} for name, Z in baselines.items()}).T
base_tbl.loc["(50 dims, no reduction)", "knn_acc_2d"] = float(cross_val_score(KNeighborsClassifier(5), X, y, cv=cv).mean())
display(base_tbl.round(4))

In [ ]:
# TODO 2（問答）
#
# (1) PCA 2D 的 trustworthiness 和 2D KNN 準確率各是多少？跟 50 維的 KNN 差多少？
#     這個差距就是「壓到 2 維損失的鄰居資訊」。你預期 t-SNE 能補回多少？
# 答：
#
# (2) PCA 2D 比隨機投影好多少？兩者都是「線性投影」，PCA 只是挑了最好的方向。
#     線性投影有什麼根本的限制？（提示：一條彎曲的帶子 / 第十四章的 moons）
# 答：

---
## 3. 手寫模型：t-SNE

| 小節 | 做什麼 | TODO |
| --- | --- | --- |
| 3A | 高維的相似度 $P$：平方距離 → 用二分搜尋找每個點的 $\sigma_i$ → 對稱化 | 3、4、5 |
| 3B | 低維的相似度 $Q$（Student-t）、KL、梯度 | 6、7 |
| 3C | 最佳化：early exaggeration、momentum、gains；在 Digits 上對照 sklearn | 9 |
| 3D | 實驗：MNIST 5k、perplexity、early exaggeration、「群的大小與距離」、隨機種子 | — |

### 3A. 高維的相似度 $P$

1. **平方距離** $d_{ij} = \lVert x_i - x_j\rVert^2$（TODO 3）。
2. **條件機率**：$p_{j\mid i} = \dfrac{\exp(-\beta_i d_{ij})}{\sum_{k \ne i}\exp(-\beta_i d_{ik})}$，$p_{i\mid i} = 0$。
   每個點的 $\beta_i$ 用**二分搜尋**，讓 $e^{H(P_i)}$ = 指定的 perplexity（TODO 4）。
   為什麼每個點不同？密集區域的點，鄰居都很近 → 需要小 $\sigma$；稀疏區域 → 大 $\sigma$。（第十四章：單一 eps 不夠用。）
3. **對稱化**：$p_{ij} = (p_{j\mid i} + p_{i\mid j}) / (2n)$（TODO 5）。全部加總 = 1；離群點 $i$ 的 $p_{j\mid i}$ 很集中，但對稱化保證每個點至少有 $\sum_j p_{ij} > 1/(2n)$ 的份量。

#### 符號表（3A）

| 符號 | 意義 | shape | 程式裡 |
| --- | --- | --- | --- |
| $d_{ij}$ | 平方距離 | $(n, n)$ | `D` |
| $d_i$ | 第 $i$ 列去掉自己 | $(n - 1,)$ | `d_i` |
| $\beta_i$ | $1/(2\sigma_i^2)$ | 純量 | `beta` |
| $p_{\cdot\mid i}$ | 第 $i$ 列的條件機率 | $(n - 1,)$ | `p` |
| $H$ | 熵（自然對數） | 純量 | `H` |
| $P$ | 對稱的聯合機率 | $(n, n)$ | `P` |

**下標公式**：
- TODO 3：$d_{ij} = \sum_l x_{il}^2 + \sum_l x_{jl}^2 - 2\sum_l x_{il}x_{jl}$ → `s[:, None] + s[None, :] - 2 * X @ X.T`。
- TODO 5：$p_{ij} = (p_{j\mid i} + p_{i\mid j}) / (2n)$ → `(Pc + Pc.T) / (2 * n)`。

#### 最小例子：TODO 4（一個點的二分搜尋）

點 $i$ 到其他 3 個點的**平方**距離 $d_i = (1, 2, 4)$，目標 perplexity = 2（「大約 2 個鄰居」）。

1. **正確答案**：試幾個 $\beta$（前置複習 1、2）：$\beta = 0.5$ → perplexity 2.59（太大、太平均）；$\beta = 1$ → 2.04；$\beta = 2$ → 1.46（太集中）。所以答案在 1 附近、略大於 1：$\beta \approx 1.047$（$\sigma \approx 0.691$），$p \approx (0.717, 0.252, 0.031)$。
2. **積木**：給定 $\beta$ 算 $p$ 和 perplexity 的函式；`lo`、`hi` 夾住答案。
3. **留給你**：寫成迴圈（`H > 目標` → $\beta$ 要變大 → `lo = beta`；反之 `hi = beta`），直到 $|H - \ln\text{Perp}| < 10^{-5}$。

In [ ]:
# 最小例子：TODO 4（一個點的二分搜尋）
d_mini = np.array([1.0, 2.0, 4.0])


def p_and_perplexity(d: np.ndarray, beta: float) -> tuple[np.ndarray, float]:
    p = np.exp(-(d - d.min()) * beta)
    p /= p.sum()
    H = -(p * np.log(np.maximum(p, 1e-300))).sum()
    return p, float(np.exp(H))


# ---- 1. 正確答案：手動試 ----
for beta in [0.5, 1.0, 1.047, 2.0]:
    p, perp = p_and_perplexity(d_mini, beta)
    print(f"beta = {beta:<5}  p = {p.round(4)}  perplexity = {perp:.4f}")

# ---- 2. 積木：H 隨 beta 遞減 ----
betas = np.logspace(-2, 1.5, 200)
fig, ax = plt.subplots(figsize=(6, 3))
ax.semilogx(betas, [p_and_perplexity(d_mini, b)[1] for b in betas])
ax.axhline(2, color="gray", ls=":")
ax.set_xlabel("beta")
ax.set_ylabel("perplexity")
plt.show()

# ---- 3. 留給你 ----
# 寫完 TODO、執行過下一格之後，取消下面的註解來對照（應為 beta ≈ 1.0474、p ≈ [0.7173 0.2517 0.031]）：
# p_m, beta_m = conditional_p_row(d_mini, perplexity=2.0); print(round(beta_m, 4), p_m.round(4))

In [ ]:
def pairwise_sq_dists(X: np.ndarray) -> np.ndarray:
    """(n, n) 的平方歐氏距離，對角線 0、沒有負數。"""
    # TODO 3（程式）：用前置複習 7 的公式；再 np.maximum(D, 0)、np.fill_diagonal(D, 0)
    raise NotImplementedError("TODO 3：平方距離")


def conditional_p_row(d_i: np.ndarray, perplexity: float, tol: float = 1e-5, max_iter: int = 100) -> tuple[np.ndarray, float]:
    """一個點到「其他所有點」的平方距離 d_i → (條件機率 p, 找到的 beta)。"""
    target = np.log(perplexity)
    lo, hi, beta = 0.0, np.inf, 1.0
    for _ in range(max_iter):
        p = np.exp(-(d_i - d_i.min()) * beta)  # 前置複習 9
        p /= p.sum()
        H = -(p * np.log(np.maximum(p, 1e-300))).sum()
        # TODO 4（程式）：二分搜尋
        #   - |H − target| < tol → break
        #   - H > target（太平均）→ beta 要變大：lo = beta；hi 還是 inf 就 beta *= 2，否則 beta = (beta + hi) / 2
        #   - H < target（太集中）→ beta 要變小：hi = beta；beta = (beta + lo) / 2
        raise NotImplementedError("TODO 4：二分搜尋")
    return p, beta


def joint_probabilities(X: np.ndarray, perplexity: float = 30.0) -> tuple[np.ndarray, np.ndarray]:
    """回傳 (對稱的 P (n, n)，每個點的 beta (n,))。"""
    n = len(X)
    D = pairwise_sq_dists(X)
    Pc = np.zeros((n, n))
    betas = np.zeros(n)
    for i in range(n):
        others = np.arange(n) != i
        Pc[i, others], betas[i] = conditional_p_row(D[i, others], perplexity)
    # TODO 5（程式）：對稱化，再把每個元素至少設成 1e-12（避免 log 0），對角線維持 0
    raise NotImplementedError("TODO 5：對稱化")
    return P, betas

**對照 sklearn**：`sklearn.manifold._t_sne._joint_probabilities(D, perplexity, verbose)` 是 sklearn 內部（底線開頭、不保證穩定）的函式，回傳壓縮成一維的上三角 $P$；`scipy.spatial.distance.squareform` 把它還原成 $(n, n)$。在 Digits 上比較（它的二分搜尋容忍度跟我們一樣是 $10^{-5}$）。

In [ ]:
from scipy.spatial.distance import squareform
from sklearn.manifold import _t_sne

Xd_df, yd_ser = load_digits(split=False)
Xd = np.asarray(Xd_df, dtype=float)
yd = np.asarray(yd_ser).astype(int)

t0 = time.perf_counter()
P_d, betas_d = joint_probabilities(Xd, perplexity=30)
print(f"Digits：P 花了 {time.perf_counter() - t0:.1f} 秒；P 的總和 = {P_d.sum():.6f}；對稱？{np.allclose(P_d, P_d.T)}")
P_sk = squareform(_t_sne._joint_probabilities(pairwise_sq_dists(Xd).astype(np.float32), 30, 0))
print("✅ 跟 sklearn 的 P 最大差異：", f"{np.abs(P_d - P_sk).max():.1e}", "（應 < 1e-8）")

sigma_d = np.sqrt(1 / (2 * betas_d))
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
axes[0].hist(sigma_d, bins=50)
axes[0].set_xlabel("sigma_i (perplexity 30)")
axes[1].scatter(np.sort(pairwise_sq_dists(Xd), axis=1)[:, 1:31].mean(axis=1) ** 0.5, sigma_d, s=4, c=yd, cmap="tab10")
axes[1].set_xlabel("mean distance to 30 nearest neighbors")
axes[1].set_ylabel("sigma_i")
plt.show()

### 3B. 低維的相似度 $Q$、KL 與梯度

**低維用 Student-t（自由度 1）**：
$$q_{ij} = \frac{(1 + \lVert y_i - y_j\rVert^2)^{-1}}{\sum_{k \ne l}(1 + \lVert y_k - y_l\rVert^2)^{-1}}$$

為什麼不用高斯？**擁擠問題（crowding problem）**：50 維裡一個點周圍可以有很多「中等距離」的點，2 維空間放不下。t 分布的重尾讓「中等距離」的點在 2 維可以被推到**更遠**而 $q$ 還不會太小 → 群與群之間拉開空白。

**損失與梯度**（推導見 van der Maaten & Hinton 2008，附錄）：
$$C = \sum_{i \ne j} p_{ij}\ln\frac{p_{ij}}{q_{ij}}, \qquad \frac{\partial C}{\partial y_i} = 4\sum_j (p_{ij} - q_{ij})\,(1 + \lVert y_i - y_j\rVert^2)^{-1}\,(y_i - y_j)$$

讀法：$p_{ij} > q_{ij}$（高維近、低維太遠）→ 梯度沿 $y_i - y_j$ 為正 → 往反方向走 = **吸引**；$p_{ij} < q_{ij}$ → **排斥**。像一組彈簧。

#### 符號表（3B）

| 符號 | 意義 | shape | 程式裡 |
| --- | --- | --- | --- |
| $Y$ | 低維座標 | $(n, 2)$ | `Y` |
| $w_{ij}$ | $(1 + \lVert y_i - y_j\rVert^2)^{-1}$，對角線 0 | $(n, n)$ | `num` |
| $Q$ | $w / \sum w$ | $(n, n)$ | `Q` |
| $M$ | $(P - Q) \odot w$ | $(n, n)$ | `M` |
| $\partial C / \partial Y$ | 梯度 | $(n, 2)$ | `grad` |

**下標公式**：
- TODO 6：$w_{ij} = 1 / (1 + d^Y_{ij})$ → `1 / (1 + pairwise_sq_dists(Y))`，對角線設 0；$q_{ij} = w_{ij} / \sum_{kl} w_{kl}$ → `num / num.sum()`。
- TODO 7：$g_i = 4\sum_j M_{ij}(y_i - y_j) = 4\big(y_i\sum_j M_{ij} - \sum_j M_{ij} y_j\big)$ → `4 * (M.sum(1)[:, None] * Y - M @ Y)`（前置複習 10）。

#### 最小例子：TODO 6、7（3 個點）

$Y = \{(0, 0), (1, 0), (0, 2)\}$；$P$ 讓點 3 跟另外兩點「很像」（0.222），點 1、2 之間「不太像」（0.056）。

1. **正確答案**：迴圈算。$w_{12} = 1/(1 + 1) = 0.5$、$w_{13} = 1/(1 + 4) = 0.2$、$w_{23} = 1/(1 + 5) = 0.167$；$Q = w / (2 \times 0.867)$ → $q_{12} = 0.288$、$q_{13} = 0.115$、$q_{23} = 0.096$；KL ≈ 0.481。
   梯度：點 1 ≈ $(0.466, -0.171)$ —— 往負梯度走 = 向左（離開點 2，因為 $q_{12} > p_{12}$）、向上（靠近點 3）；點 3 ≈ $(-0.084, 0.339)$ → 往下靠近另外兩點。
2. **積木**：`pairwise_sq_dists`（TODO 3）、`M.sum(1)`、`M @ Y`。
3. **留給你**：向量化。

In [ ]:
# 最小例子：TODO 6、7（3 個點）
Y_mini = np.array([[0.0, 0.0], [1.0, 0.0], [0.0, 2.0]])
P_mini = np.array([[0, 1, 4], [1, 0, 4], [4, 4, 0]]) / 18

# ---- 1. 正確答案：迴圈 ----
w_loop = np.zeros((3, 3))
for i in range(3):
    for j in range(3):
        if i != j:
            w_loop[i, j] = 1 / (1 + ((Y_mini[i] - Y_mini[j]) ** 2).sum())
Q_loop = w_loop / w_loop.sum()
mask_mini = ~np.eye(3, dtype=bool)
kl_loop = (P_mini[mask_mini] * np.log(P_mini[mask_mini] / Q_loop[mask_mini])).sum()
g_loop = np.zeros((3, 2))
for i in range(3):
    for j in range(3):
        if i != j:
            g_loop[i] += 4 * (P_mini[i, j] - Q_loop[i, j]) * w_loop[i, j] * (Y_mini[i] - Y_mini[j])
print("Q =\n", Q_loop.round(4), "\nKL =", kl_loop.round(4), "\ngrad =\n", g_loop.round(4))

# ---- 2. 積木 ----
M_mini = (P_mini - Q_loop) * w_loop
print("\nM.sum(1) =", M_mini.sum(1).round(4), "；M @ Y =\n", (M_mini @ Y_mini).round(4))

# ---- 3. 留給你 ----
# 寫完 TODO、執行過下一格之後，取消下面的註解來對照（應跟上面的 KL、grad 一樣）：
# print(round(kl_and_grad(Y_mini, P_mini)[0], 4), kl_and_grad(Y_mini, P_mini)[1].round(4))

In [ ]:
def kl_and_grad(Y: np.ndarray, P: np.ndarray, compute_kl: bool = True) -> tuple[float, np.ndarray]:
    """回傳 (KL(P‖Q), ∂KL/∂Y (n, 2))。compute_kl=False 時 KL 回傳 nan（省一次 n × n 的 log，最佳化時用）。"""
    # TODO 6（程式）：num（對角線 0）與 Q；Q 至少 1e-12
    raise NotImplementedError("TODO 6：Q")
    kl = float((P * np.log(np.maximum(P, 1e-300) / Q)).sum()) if compute_kl else np.nan  # p = 0 的項是 0
    # TODO 7（程式）：M = (P − Q) * num；grad = 4 * (M.sum(1)[:, None] * Y − M @ Y)
    raise NotImplementedError("TODO 7：梯度")
    return kl, grad


# 對照 sklearn 內部的 _kl_divergence（exact 版的損失與梯度），以及數值微分
Y0 = np.random.default_rng(0).normal(size=(len(Xd), 2))
kl_m, g_m = kl_and_grad(Y0, P_d)
kl_s, g_s = _t_sne._kl_divergence(Y0.ravel(), squareform(P_d, checks=False), 1, len(Xd), 2, compute_error=True)
print(f"✅ KL：手寫 {kl_m:.10f}、sklearn {kl_s:.10f}；梯度最大差異 {np.abs(g_m.ravel() - g_s).max():.1e}（梯度最大值 {np.abs(g_s).max():.1e}）")
eps, (i0, k0) = 1e-6, (5, 1)
Yp, Ym = Y0.copy(), Y0.copy()
Yp[i0, k0] += eps
Ym[i0, k0] -= eps
num_grad = (kl_and_grad(Yp, P_d)[0] - kl_and_grad(Ym, P_d)[0]) / (2 * eps)
print(f"✅ 數值微分：(KL(y+ε) − KL(y−ε)) / 2ε = {num_grad:.6e}；解析梯度 = {g_m[i0, k0]:.6e}")

In [ ]:
# TODO 8（問答）TODO 3–7
#
# (1) 最小例子：
#     (a) 二分搜尋的 beta 為什麼一開始用「乘 2」而不是「取中點」？（提示：hi 一開始是 ∞）
#     (b) 3 個點的例子：點 1、2 會互相吸引還是排斥？用 p_12 和 q_12 的大小解釋。
# 答：
#
# (2) σ_i 的分布：σ 最大的點在哪裡（密集還是稀疏的區域）？為什麼 t-SNE 要讓每個點有自己的 σ？
#     如果全部用同一個 σ，稀疏區域的點會怎樣？（對照第十四章：單一 eps 的問題）
# 答：
#
# (3) KL 的不對稱（前置複習 3）：
#     (a) 高維很近（p 大）但低維很遠（q 小）：每一對的懲罰大還是小？
#     (b) 高維很遠（p ≈ 0）但低維很近（q 大）：懲罰大還是小？
#     (c) 所以 t-SNE 圖上「兩群靠得很近」能不能解讀成「這兩群在高維也很近」？
# 答：

### 3C. 最佳化

t-SNE 的損失不是凸函數，最佳化的細節（Maaten & Hinton 2008；sklearn 的預設）對結果影響很大：

| 技巧 | 做法 | 為什麼 |
| --- | --- | --- |
| 初始化 | PCA 前 2 維，縮放到標準差 $10^{-4}$（`init="pca"`）；或隨機 | 從一個保留全域結構的起點開始；很小 → 一開始所有 $q$ 差不多 |
| early exaggeration | 前 250 步把 $P$ 乘以 12 | 吸引力變強 → 群先各自聚緊、移動到大致的位置，之後再微調 |
| momentum | 前 250 步 0.5、之後 0.8 | 前置複習 6 |
| gains（delta-bar-delta） | 梯度方向跟上一步更新**相反** → 這一維的步長 +0.2；**相同** → ×0.8；最小 0.01 | 每個座標自己調步長 |
| 學習率 | $\max(n / 12 / 4,\ 50)$（sklearn 的 `learning_rate="auto"`） | 點越多、步伐越大 |

#### 符號表（3C）

| 符號 | 意義 | 程式裡 |
| --- | --- | --- |
| $\Delta$ | 上一步的位移 | `update` |
| gains | 每個座標的步長倍數 | `gains` |
| $\alpha$ | exaggeration（前 250 步 12，之後 1） | `exag` |
| $\eta$ | 學習率 | `lr` |

In [ ]:
from typing import Literal, Self


class TSNEScratch:
    """exact t-SNE（O(n²) 記憶體與時間）。屬性名稱仿 sklearn：embedding_、kl_divergence_。"""

    def __init__(self, perplexity: float = 30.0, n_iter: int = 1000, early_exaggeration: float = 12.0,
                 exaggeration_iter: int = 250, init: Literal["pca", "random"] = "pca", random_state: int = 0) -> None:
        self.perplexity = perplexity
        self.n_iter = n_iter
        self.early_exaggeration = early_exaggeration
        self.exaggeration_iter = exaggeration_iter
        self.init = init
        self.random_state = random_state

    def fit(self, X: np.ndarray, P: np.ndarray | None = None) -> Self:
        """P 可以事先算好傳進來（實驗時重複使用，省時間）。"""
        X = np.asarray(X, dtype=float)
        n = len(X)
        self.P_ = joint_probabilities(X, self.perplexity)[0] if P is None else P
        if self.init == "pca":
            Y = PCA(2, svd_solver="full").fit_transform(X)
            Y = Y / Y[:, 0].std() * 1e-4
        else:
            Y = np.random.default_rng(self.random_state).normal(size=(n, 2)) * 1e-4
        lr = max(n / self.early_exaggeration / 4, 50)
        update = np.zeros_like(Y)
        gains = np.ones_like(Y)
        self.history_ = []
        for it in range(self.n_iter):
            early = it < self.exaggeration_iter
            exag = self.early_exaggeration if early else 1.0
            momentum = 0.5 if early else 0.8
            _, grad = kl_and_grad(Y, self.P_ * exag, compute_kl=False)
            # TODO 9（程式）：一步更新
            #   - inc = update * grad < 0（方向相反）→ gains[inc] += 0.2；gains[~inc] *= 0.8；再 np.clip(gains, 0.01, None)
            #   - update = momentum * update − lr * gains * grad
            #   - Y = Y + update
            raise NotImplementedError("TODO 9：更新")
            if it % 50 == 0 or it == self.n_iter - 1:
                self.history_.append((it, kl_and_grad(Y, self.P_)[0]))  # 記錄「沒有放大」的 KL
        self.embedding_ = Y
        self.kl_divergence_ = kl_and_grad(Y, self.P_)[0]
        return self

    def fit_transform(self, X: np.ndarray, P: np.ndarray | None = None) -> np.ndarray:
        return self.fit(X, P).embedding_

**在 Digits 上對照 sklearn**：`TSNE(method="exact")` 用的就是同一個演算法（同樣的 $P$、梯度、gains、momentum、exaggeration）。差別：sklearn 會在梯度很小時提早停、PCA 初始化的正負號可能不同 → 座標不會一模一樣，但最後的 KL 與 trustworthiness 應該很接近。`method="barnes_hut"`（預設）用四元樹近似排斥力，快很多。

In [ ]:
from sklearn.manifold import TSNE

t0 = time.perf_counter()
ts_d = TSNEScratch(perplexity=30).fit(Xd, P=P_d)
t_mine = time.perf_counter() - t0
rows = [{"model": "TSNEScratch (exact)", "KL": ts_d.kl_divergence_, **trustworthiness(Xd, ts_d.embedding_), "seconds": t_mine}]
emb_d = {"TSNEScratch (exact)": ts_d.embedding_}
for method in ["exact", "barnes_hut"]:
    t0 = time.perf_counter()
    sk = TSNE(2, perplexity=30, method=method, init="pca", random_state=0)
    Z = sk.fit_transform(Xd)
    rows.append({"model": f"sklearn TSNE ({method})", "KL": sk.kl_divergence_, **trustworthiness(Xd, Z), "seconds": time.perf_counter() - t0})
    emb_d[f"sklearn TSNE ({method})"] = Z
display(pd.DataFrame(rows).set_index("model").round(4))

fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
hist = np.array(ts_d.history_)
axes[0].plot(hist[:, 0], hist[:, 1], "o-", ms=3)
axes[0].axvline(250, color="gray", ls=":")
axes[0].set_xlabel("iteration")
axes[0].set_ylabel("KL(P || Q)")
axes[0].set_title("TSNEScratch on Digits")
for ax, (name, Z) in zip(axes[1:], emb_d.items()):
    ax.scatter(Z[:, 0], Z[:, 1], c=yd, cmap="tab10", s=4)
    for c in range(10):
        ax.text(*np.median(Z[yd == c], axis=0), str(c), fontsize=13, weight="bold")
    ax.set_title(name)
plt.show()

### 3D. 實驗

| 實驗 | 問題 |
| --- | --- |
| 1 | 手寫 t-SNE 跑 MNIST 5k：要多久？跟 PCA 2D 比分得多開？ |
| 2 | perplexity 5、30、100 有什麼不同？ |
| 3 | 不做 early exaggeration、或隨機初始化會怎樣？ |
| 4 | **群的大小與距離**：已知真相的合成資料 —— t-SNE 會保留「群的大小」和「群之間的距離」嗎？ |
| 5 | 不同隨機種子的結果一樣嗎？ |

#### 本節名詞

| 名詞 | 意思 |
| --- | --- |
| 合成資料（實驗 4） | 3 群 10 維高斯：A、B 的標準差 1、C 的標準差 5（大 10 倍面積）；A–B 中心距離 10、A–C 距離 50 |

In [ ]:
# 實驗 1：手寫 t-SNE 跑 MNIST 5k（每一步要算好幾個 5,000 × 5,000 的矩陣，整格大約 5–10 分鐘）
t0 = time.perf_counter()
P_m, betas_m = joint_probabilities(X, perplexity=30)
t_P = time.perf_counter() - t0
t0 = time.perf_counter()
ts_m = TSNEScratch(perplexity=30).fit(X, P=P_m)
t_opt = time.perf_counter() - t0
Z_tsne = ts_m.embedding_
print(f"P：{t_P:.0f} 秒；1000 步最佳化：{t_opt:.0f} 秒（每步 {t_opt / 1000 * 1000:.0f} ms）；最後 KL = {ts_m.kl_divergence_:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))
for ax, (name, Z) in zip(axes, [("PCA 2D", Z_pca), ("TSNEScratch (perplexity 30)", Z_tsne)]):
    ax.scatter(Z[:, 0], Z[:, 1], c=y, cmap="tab10", s=3)
    for c in range(10):
        ax.text(*np.median(Z[y == c], axis=0), str(c), fontsize=15, weight="bold")
    ax.set_title(f"MNIST 5k: {name}")
plt.show()
print({"PCA 2D": {**trustworthiness(X, Z_pca), "knn_acc_2d": round(knn_acc_2d(Z_pca, y), 4)},
       "TSNEScratch": {**trustworthiness(X, Z_tsne), "knn_acc_2d": round(knn_acc_2d(Z_tsne, y), 4)}})

In [ ]:
# 實驗 2：perplexity（用 sklearn 的 Barnes-Hut 版，比較快）
perps = [5, 30, 100]
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
rows = []
for ax, perp in zip(axes, perps):
    Z = TSNE(2, perplexity=perp, init="pca", random_state=0).fit_transform(X)
    ax.scatter(Z[:, 0], Z[:, 1], c=y, cmap="tab10", s=3)
    ax.set_title(f"perplexity = {perp}")
    rows.append({"perplexity": perp, **trustworthiness(X, Z), "trust (k=50)": trustworthiness(X, Z, n_neighbors=50)["trustworthiness"], "knn_acc_2d": knn_acc_2d(Z, y)})
plt.show()
display(pd.DataFrame(rows).set_index("perplexity").round(4))

In [ ]:
# 實驗 3：early exaggeration 與初始化（手寫版，Digits；重複使用 P_d）
from sklearn.cluster import DBSCAN


def n_pieces(Z: np.ndarray, members: np.ndarray, min_frac: float = 0.1) -> int:
    """一個數字在 2D 上被拆成幾塊：對它的點做 DBSCAN（eps = 整張圖「第 5 近鄰距離」中位數的 3 倍），數佔 ≥ 10% 的群。"""
    eps = 3 * np.median(NearestNeighbors(n_neighbors=6).fit(Z).kneighbors(Z)[0][:, 5])
    lab = DBSCAN(eps=eps, min_samples=5).fit_predict(Z[members])
    return int(sum((lab == k).mean() >= min_frac for k in set(lab) - {-1}))


settings = {
    "default (exag 12, PCA init)": dict(),
    "no exaggeration": dict(early_exaggeration=1.0),
    "random init": dict(init="random"),
    "random init, no exaggeration": dict(init="random", early_exaggeration=1.0),
}
fig, axes = plt.subplots(1, 4, figsize=(22, 4.8))
rows = []
for ax, (name, kw) in zip(axes, settings.items()):
    m = TSNEScratch(**kw).fit(Xd, P=P_d)
    Z = m.embedding_
    ax.scatter(Z[:, 0], Z[:, 1], c=yd, cmap="tab10", s=4)
    for c in range(10):
        ax.text(*np.median(Z[yd == c], axis=0), str(c), fontsize=12, weight="bold")
    ax.set_title(name)
    rows.append({"setting": name, "KL": m.kl_divergence_, **trustworthiness(Xd, Z), "knn_acc_2d": knn_acc_2d(Z, yd),
                 "# digits split into >1 piece": sum(n_pieces(Z, yd == c) > 1 for c in range(10))})
plt.show()
display(pd.DataFrame(rows).set_index("setting").round(4))

In [ ]:
# 實驗 4：群的大小與群之間的距離（已知真相的合成資料）
rng = np.random.default_rng(0)
centers = np.zeros((3, 10))
centers[1, 0] = 10   # B 離 A 10
centers[2, 0] = 50   # C 離 A 50
spreads = [1.0, 1.0, 5.0]
Xs = np.vstack([rng.normal(centers[k], spreads[k], size=(300, 10)) for k in range(3)])
ys = np.repeat([0, 1, 2], 300)


def summarize(Z: np.ndarray, labels: np.ndarray) -> dict:
    cent = np.array([Z[labels == k].mean(axis=0) for k in range(3)])
    size = [float(np.sqrt(((Z[labels == k] - cent[k]) ** 2).sum(1).mean())) for k in range(3)]
    return {"size A": size[0], "size B": size[1], "size C": size[2],
            "dist A–B": float(np.linalg.norm(cent[0] - cent[1])), "dist A–C": float(np.linalg.norm(cent[0] - cent[2]))}


fig, axes = plt.subplots(1, 4, figsize=(22, 4.5))
rows = {"high-dim (truth)": summarize(Xs, ys), "PCA 2D": summarize(PCA(2).fit_transform(Xs), ys)}
for ax, perp in zip(axes[1:], [5, 30, 200]):
    Z = TSNE(2, perplexity=perp, init="pca", random_state=0).fit_transform(Xs)
    rows[f"t-SNE perp {perp}"] = summarize(Z, ys)
    ax.scatter(Z[:, 0], Z[:, 1], c=ys, cmap="brg", s=5)
    ax.set_title(f"t-SNE perplexity {perp}")
Zp = PCA(2).fit_transform(Xs)
axes[0].scatter(Zp[:, 0], Zp[:, 1], c=ys, cmap="brg", s=5)
axes[0].set_title("PCA 2D (A=blue, B=red, C=green)")
plt.show()
tbl4 = pd.DataFrame(rows).T
tbl4["size C / size A"] = tbl4["size C"] / tbl4["size A"]
tbl4["dist A–C / dist A–B"] = tbl4["dist A–C"] / tbl4["dist A–B"]
display(tbl4.round(2))

In [ ]:
# 實驗 5：隨機種子（隨機初始化；sklearn Barnes-Hut；MNIST 5k）
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for ax, seed in zip(axes, [0, 1, 2]):
    Z = TSNE(2, init="random", random_state=seed).fit_transform(X)
    ax.scatter(Z[:, 0], Z[:, 1], c=y, cmap="tab10", s=3)
    for c in range(10):
        ax.text(*np.median(Z[y == c], axis=0), str(c), fontsize=13, weight="bold")
    ax.set_title(f"random init, seed {seed}  (trust {trustworthiness(X, Z)['trustworthiness']:.3f})")
plt.show()

In [ ]:
# TODO 10（問答）實驗 1–5
#
# (1) 實驗 1：
#     (a) P 和 1000 步各花多久？如果 n 變 2 倍（10k），時間大約變幾倍？記憶體呢？
#     (b) t-SNE 圖上，PCA 2D 擠在一起的數字（4、7、9；3、5、8）分開了嗎？trustworthiness、2D KNN 準確率各提升多少？
#     (c) 有沒有「跑錯群」的點（例如一小撮 4 在 9 的群裡）？第 6 節會把它們找出來。
# 答：
#
# (2) 實驗 2：
#     (a) perplexity 5：群是一大塊、還是碎成很多小塊？為什麼？（提示：每個點只「看」5 個鄰居）
#     (b) perplexity 100：群之間的空白變大還是變小？trust (k=5) 和 trust (k=50) 誰升誰降？
#     (c) perplexity 跟第四章 KNN 的 K、第十四章 DBSCAN 的 min_samples 有什麼相似之處？
# 答：
#
# (3) 實驗 3：拿掉 early exaggeration 或改成隨機初始化，KL、trustworthiness、被拆成兩塊的數字個數（表格最後一欄）差多少？
#     early exaggeration 的設計目的是「一開始吸引力 × 12 → 同一群的點先抱在一起、整群移動到大致的位置」。
#     在 Digits（1,797 點、群分得很開）上它的效果明顯嗎？你預期在什麼情況下它比較重要？（提示：點很多、隨機初始化、群跟群靠得很近）
#     哪兩個數字被拆成兩塊？看圖：是 t-SNE 的問題，還是那個數字本來就有兩種寫法？
# 答：
#
# (4) 實驗 4（最重要）：
#     (a) 高維中 C 的大小是 A 的 5 倍。t-SNE 圖上呢？三種 perplexity 都一樣嗎？
#     (b) 高維中 A–C 的距離是 A–B 的 5 倍。t-SNE 圖上呢？PCA 圖上呢？
#     (c) 結論：讀 t-SNE 圖時，「群的大小」「群之間的距離」能不能當真？什麼可以當真？
#         （提示：t-SNE 每個點有自己的 σ —— 稀疏的群會被「放大」到跟密集的群差不多大）
# 答：
#
# (5) 實驗 5：不同種子下，數字的「相對位置」（誰在誰旁邊）一樣嗎？「哪些點在同一群」一樣嗎？
#     如果要在報告裡放 t-SNE 圖，你會怎麼做才不會誤導讀者？
# 答：

---
## 4. 套件實作：sklearn `TSNE` 與 `umap-learn`

#### sklearn `TSNE`

| 參數 | 意思 |
| --- | --- |
| `perplexity` | 有效鄰居數（5–50 常見；要小於 $n$） |
| `method` | `"barnes_hut"`（預設，$O(n\log n)$，只能 2–3 維）、`"exact"`（$O(n^2)$） |
| `angle` | Barnes-Hut 的近似程度（0.5；越大越快越不準） |
| `init` | `"pca"`（預設）或 `"random"` |
| `learning_rate` | `"auto"` = $\max(n / 12 / 4, 50)$ |
| `early_exaggeration` | 12 |
| `max_iter` | 1000 |

**t-SNE 沒有 `transform`**：新的點進來要整個重跑（嵌入只對訓練的點有定義）。

#### UMAP（McInnes et al. 2018）

| 步驟 | t-SNE | UMAP |
| --- | --- | --- |
| 高維相似度 | 所有點的高斯，perplexity 定 $\sigma_i$ | 只看 `n_neighbors` 個最近鄰居（近似 kNN 圖）；$\exp(-(d - \rho_i)/\sigma_i)$，$\rho_i$ = 到最近鄰居的距離 |
| 對稱化 | 平均 | 模糊聯集 $a + b - ab$ |
| 低維相似度 | Student-t | $(1 + a\,d^{2b})^{-1}$，$a$、$b$ 由 `min_dist` 決定 |
| 損失 | KL（只懲罰「鄰居被拆散」） | 交叉熵（也懲罰「非鄰居被放太近」）→ 保留較多全域結構 |
| 最佳化 | 全部 $n^2$ 配對 | SGD + 負採樣（只抽少數非鄰居來推開） |
| 新資料 | 不行 | `transform` |

| 參數 | 意思 |
| --- | --- |
| `n_neighbors` | 15（預設）；越大越看全域結構，類似 perplexity |
| `min_dist` | 0.1（預設）；低維中點最近可以靠多近 —— 越小群越緊 |
| `metric` | 距離（`"euclidean"`、`"cosine"`…） |

In [ ]:
import warnings

import umap

warnings.filterwarnings("ignore", message=".*n_jobs value.*")  # umap 設 random_state 時會關掉平行，提醒一次就好

speed_rows = []
embeddings = {"PCA 2D": Z_pca, "TSNEScratch (exact)": Z_tsne}
for name, make in {
    "sklearn TSNE (barnes_hut)": lambda: TSNE(2, init="pca", random_state=0),
    "UMAP (n_neighbors 15, min_dist 0.1)": lambda: umap.UMAP(random_state=0),
}.items():
    t0 = time.perf_counter()
    embeddings[name] = make().fit_transform(X)
    speed_rows.append({"model": name, "seconds": time.perf_counter() - t0})
speed_rows.append({"model": "TSNEScratch (exact)", "seconds": t_P + t_opt})
display(pd.DataFrame(speed_rows).set_index("model").round(1))

fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))
for ax, name in zip(axes, ["sklearn TSNE (barnes_hut)", "UMAP (n_neighbors 15, min_dist 0.1)"]):
    Z = embeddings[name]
    ax.scatter(Z[:, 0], Z[:, 1], c=y, cmap="tab10", s=3)
    for c in range(10):
        ax.text(*np.median(Z[y == c], axis=0), str(c), fontsize=15, weight="bold")
    ax.set_title(name)
plt.show()

In [ ]:
# UMAP 的兩個超參數
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
for r, nn_ in enumerate([5, 50]):
    for c, md_ in enumerate([0.0, 0.1, 0.8]):
        Z = umap.UMAP(n_neighbors=nn_, min_dist=md_, random_state=0).fit_transform(X)
        axes[r, c].scatter(Z[:, 0], Z[:, 1], c=y, cmap="tab10", s=2)
        axes[r, c].set_title(f"n_neighbors={nn_}, min_dist={md_}")
plt.show()

# UMAP 可以 transform 新的點：用前 4,000 張 fit，把後 1,000 張投影上去
um = umap.UMAP(random_state=0).fit(X[:4000])
Z_new = um.transform(X[4000:])
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(um.embedding_[:, 0], um.embedding_[:, 1], c=y[:4000], cmap="tab10", s=2, alpha=0.3)
ax.scatter(Z_new[:, 0], Z_new[:, 1], c=y[4000:], cmap="tab10", s=12, marker="x")
ax.set_title("UMAP: fit on 4,000 (dots), transform 1,000 new (x)")
plt.show()
knn_new = KNeighborsClassifier(5).fit(um.embedding_, y[:4000])
print(f"新點在 2D 上用 5-NN 分類（只靠 transform 出來的座標）：accuracy = {knn_new.score(Z_new, y[4000:]):.4f}")

In [ ]:
# TODO 11（問答）
#
# (1) 速度：手寫 exact、sklearn Barnes-Hut、UMAP 各花多久？手寫版主要慢在哪兩步？
#     如果是全部 70,000 張，哪一個還跑得動？
# 答：
#
# (2) UMAP 和 t-SNE 的圖有什麼不同？（群的緊密程度、群之間的空白、相似數字的相對位置）
#     用上面的表格解釋：UMAP 的損失也懲罰「非鄰居被放太近」，這對全域結構有什麼影響？
# 答：
#
# (3) UMAP 的超參數：
#     (a) n_neighbors 5 vs 50：哪一個的群比較碎？跟實驗 2 的 perplexity 比。
#     (b) min_dist 0.0 vs 0.8：群的「緊密度」差多少？如果要拿 UMAP 的輸出去做分群（第十四章 DBSCAN），你會選哪個？
# 答：
#
# (4) transform：為什麼 t-SNE 不能 transform 新的點，UMAP 可以？新點投影的準確率跟訓練點比如何？
# 答：

---
## 5. 評估：一張降維圖好不好？

降維沒有「正確答案」，常用三種角度（都只看鄰居，不看標籤；標籤只用在第 3 種）：

| 指標 | 定義 | 看什麼 |
| --- | --- | --- |
| trustworthiness | 低維的 $k$ 個鄰居，在高維排名多前面（有假鄰居就扣分） | 低維的鄰居可信嗎 |
| **鄰居重疊率**（kNN overlap，TODO 12） | 高維的 $k$ 近鄰與低維的 $k$ 近鄰，交集佔幾成 | 高維的鄰居保留了多少 |
| 2D KNN 準確率 | 只用低維座標做 KNN 分類 | 同類有沒有聚在一起 |
| 群中心距離的相關 | 高維與低維的「數字中心之間的距離」的 Spearman 相關 | 全域結構（群的相對位置） |

#### 最小例子：TODO 12（鄰居重疊率）

一維的 5 個點，高維座標 `[0, 1, 2, 10, 11]`；低維座標 `[0, 2, 1, 11, 10]`（0、1、2 這群裡的順序被打亂了，但還是一群）。$k = 1$。

1. **正確答案**：高維最近鄰：0→1、1→0（或 2，距離相同；`NearestNeighbors` 取索引小的 0）、2→1、10→11、11→10。低維最近鄰（座標 0→2、2→1、1→0 …）：點 0（座標 0）→ 點 2（座標 1）；點 1（座標 2）→ 點 2（座標 1）；點 2（座標 1）→ 點 0 或 1（距離相同，取 0）；點 3 → 點 4；點 4 → 點 3。對得上：點 3、4 → 2/5 = 0.4。
2. **積木**：`NearestNeighbors(n_neighbors=k + 1).fit(X).kneighbors(X, return_distance=False)[:, 1:]`（第一個是自己）、`np.intersect1d`。
3. **留給你**：每一列算交集大小 / k，再平均。

In [ ]:
# 最小例子：TODO 12（鄰居重疊率）
from sklearn.neighbors import NearestNeighbors

Xh_mini = np.array([[0.0], [1.0], [2.0], [10.0], [11.0]])
Xl_mini = np.array([[0.0], [2.0], [1.0], [11.0], [10.0]])
# ---- 2. 積木 ----
nh = NearestNeighbors(n_neighbors=2).fit(Xh_mini).kneighbors(Xh_mini, return_distance=False)[:, 1:]
nl = NearestNeighbors(n_neighbors=2).fit(Xl_mini).kneighbors(Xl_mini, return_distance=False)[:, 1:]
print("高維最近鄰：", nh.ravel(), "；低維最近鄰：", nl.ravel())
# ---- 1. 正確答案：迴圈 ----
print("迴圈：", np.mean([len(np.intersect1d(nh[i], nl[i])) / 1 for i in range(5)]))
# ---- 3. 留給你 ----
# 寫完 TODO、執行過下一格之後，取消下面的註解來對照（應為 0.4）：
# print(knn_overlap(Xh_mini, Xl_mini, k=1))

In [ ]:
from scipy.stats import spearmanr
from ml_dl_llm.results import compare, log_result


def knn_overlap(X_high: np.ndarray, X_low: np.ndarray, k: int = 10) -> float:
    """高維 k 近鄰與低維 k 近鄰的平均交集比例（0–1）。"""
    nh = NearestNeighbors(n_neighbors=k + 1).fit(X_high).kneighbors(X_high, return_distance=False)[:, 1:]
    nl = NearestNeighbors(n_neighbors=k + 1).fit(X_low).kneighbors(X_low, return_distance=False)[:, 1:]
    # TODO 12（程式）：每一列 len(np.intersect1d(nh[i], nl[i])) / k，再平均（回傳 float）
    raise NotImplementedError("TODO 12：鄰居重疊率")


def centroid_dist_corr(X_high: np.ndarray, X_low: np.ndarray, labels: np.ndarray) -> float:
    """數字中心之間的距離（45 對），高維 vs 低維的 Spearman 相關。"""
    ch = np.array([X_high[labels == c].mean(0) for c in np.unique(labels)])
    cl = np.array([X_low[labels == c].mean(0) for c in np.unique(labels)])
    iu = np.triu_indices(len(ch), 1)
    return float(spearmanr(np.sqrt(pairwise_sq_dists(ch))[iu], np.sqrt(pairwise_sq_dists(cl))[iu])[0])


def embedding_scores(X_high: np.ndarray, Z: np.ndarray, labels: np.ndarray) -> dict:
    return {**trustworthiness(X_high, Z), "knn_overlap_10": round(knn_overlap(X_high, Z, 10), 4),
            "knn_acc_2d": round(knn_acc_2d(Z, labels), 4), "centroid_dist_spearman": round(centroid_dist_corr(X_high, Z, labels), 4)}


BENCH = "mnist5k_embedding"
scores = {}
for name, Z in embeddings.items():
    scores[name] = embedding_scores(X, Z, y)
    log_result(BENCH, name, scores[name], n=n, input="PCA-50")
display(pd.DataFrame(scores).T)
compare(BENCH, "trustworthiness")

**接續第十六章的 `digits_embedding`**：同一份 Digits，把 t-SNE（手寫）、UMAP 加進第十六章的表格（PCA k = 2、隨機投影 k = 2 已經在裡面）。注意：第十六章是在 64 個原始像素上量 trustworthiness，這裡也一樣。

In [ ]:
BENCH_D = "digits_embedding"
digits_runs = {
    "t-SNE perp 30 (scratch)": ts_d.embedding_,
    "UMAP (n_neighbors 15, min_dist 0.1)": umap.UMAP(random_state=0).fit_transform(Xd),
}
for name, Z in digits_runs.items():
    log_result(BENCH_D, name, {**trustworthiness(Xd, Z), "knn_overlap_10": round(knn_overlap(Xd, Z, 10), 4)}, k=2)
compare(BENCH_D, "trustworthiness")

In [ ]:
# TODO 13（問答）評估
#
# (1) 四個指標各是哪個方法最好？有沒有「t-SNE 在局部指標贏、PCA 在全域指標（centroid_dist_spearman）贏」的情況？
# 答：
#
# (2) knn_overlap_10 只有零點幾 —— 是不是代表 t-SNE「很差」？2 維要完美保留 50 維的 10 個近鄰可能嗎？
#     你會怎麼用這個數字「比較」不同方法，而不是看它的絕對值？
# 答：
#
# (3) digits_embedding：t-SNE、UMAP 的 trustworthiness 比第十六章 PCA k = 2、k = 10、k = 21 高還是低？
#     「2 維的 t-SNE 比 21 維的 PCA 還可信」這句話合理嗎？為什麼？
# 答：

---
## 6. 診斷：跑錯群的點

在 t-SNE 圖上用 5-NN（2D 座標）預測每個點的標籤，**預測 ≠ 真實標籤**的點 = 落在別的數字群裡的點。把它們畫出來：是 t-SNE 弄錯，還是影像本身就像別的數字（或標錯）？

In [ ]:
from sklearn.model_selection import cross_val_predict

pred_2d = cross_val_predict(KNeighborsClassifier(5), Z_tsne, y, cv=cv)
pred_50 = cross_val_predict(KNeighborsClassifier(5), X, y, cv=cv)
wrong = np.flatnonzero(pred_2d != y)
print(f"t-SNE 2D 上跑錯群的點：{len(wrong)} 個（{len(wrong) / n:.1%}）；其中在 50 維 KNN 也錯的：{int((pred_50[wrong] != y[wrong]).sum())} 個")
conf = pd.crosstab(pd.Series(y[wrong], name="true"), pd.Series(pred_2d[wrong], name="neighbors in t-SNE"))
display(conf)

show = wrong[:16]
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for ax, i in zip(axes.ravel(), show):
    ax.imshow(X_raw[i].reshape(28, 28), cmap="gray_r")
    ax.set_title(f"true {y[i]} → near {pred_2d[i]}", fontsize=9)
    ax.axis("off")
plt.show()

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(Z_tsne[:, 0], Z_tsne[:, 1], c="lightgray", s=3)
ax.scatter(Z_tsne[wrong, 0], Z_tsne[wrong, 1], c=y[wrong], cmap="tab10", s=18, edgecolor="k", linewidth=0.3)
ax.set_title("misplaced points (colored by true label)")
plt.show()

In [ ]:
# TODO 14（問答，本章總結）
#
# (1) 跑錯群的點：
#     (a) 最常見的「true → near」是哪幾對？跟 PCA 2D 擠在一起的數字一樣嗎？
#     (b) 這些點在 50 維的 KNN 也錯嗎？所以錯誤主要來自「降維」還是「資料本身」？
#     (c) 看影像：有沒有你自己也認不出來、或懷疑標錯的？
# 答：
#
# (2) 什麼時候用哪一個？
#     |                    | PCA | t-SNE | UMAP |
#     | 線性／非線性        |     |       |      |
#     | 保留什麼            |     |       |      |
#     | 可以 transform 新點 |     |       |      |
#     | 速度（n = 70k）     |     |       |      |
#     | 圖上的距離能不能讀  |     |       |      |
#     | 適合當下游模型的輸入 |     |       |      |
# 答：
#
# (3) 「t-SNE 圖上分成 10 群，所以資料有 10 類」—— 這個推論哪裡有問題？（實驗 2、4、5）
#     要確認資料真的有群結構，你會用第十一～十三章的哪些方法、在哪個空間（高維還是 t-SNE 的 2D）做？
# 答：
#
# (4) 下一章 HMM 換到「序列」：資料不再是獨立的點，而是有順序。PCA／t-SNE 會把一段時間序列的每個時間點當成獨立的點 —— 會丟掉什麼？
# 答：

---
## 7. 本章小結

（完成所有 TODO、並經過回饋與修正之後，依 README「章末小結」填寫：問答註解轉成 markdown、結果表、核心觀念、待補強。）

### 方法與結果

MNIST 5k（PCA 50 維後；標籤只用於上色與評估）。

| 方法 | 做法 | 關鍵性質 | trustworthiness | knn_overlap_10 | 2D KNN acc | 秒數 |
| --- | --- | --- | --- | --- | --- | --- |
| PCA 2D | 線性投影 | | | | | |
| t-SNE（手寫，exact） | | | | | | |
| t-SNE（sklearn，Barnes-Hut） | | | | | | |
| UMAP | | | | | | |

### 核心觀念

1.
2.
3.

### 待補強（之後回來可以學）

**理論**
- ☐ t-SNE 梯度的完整推導；SNE（對稱前的版本）與擁擠問題的分析
- ☐ Barnes-Hut 近似（四元樹）與 FFT 加速的 FIt-SNE
- ☐ UMAP 的拓撲背景（模糊單純複形、黎曼流形假設）
- ☐ t-SNE 的理論保證（Linderman & Steinerberger 2019：early exaggeration 與譜分群的關係）

**延伸**
- ☐ openTSNE（可以 transform 新點的 t-SNE 實作）；多尺度 perplexity
- ☐ 其他非線性降維：Isomap、LLE、Laplacian Eigenmaps、PaCMAP、TriMap
- ☐ 用 UMAP 的輸出 + HDBSCAN 分群（第十四章）
- ☐ 自編碼器（第四部分 Autoencoder）：可以 transform、可以重建的非線性降維
- ☐ 把 BERT、CLIP 的 embedding 用 UMAP 視覺化（第五部分）